# PPO Clipped Policy Losss
**Core problem:** make policy improvement whithin a "trust region" using clip to approximate the KL constraint so that a single update can reuse old data (importance sampling) without blowing up when the ratio is large.

\begin{equation}
L^{CLIP} = - min(r_{t}(\theta).A_{t},clip(r_{t}(\theta),1-\epsilon,1+\epsilon).A_{t})
\end{equation}

\begin{equation}
r_{t}(\theta) = \pi_{policy}(a_{t}/s_{t}) / \pi_{old}(a_{t}/s_{t})
\end{equation}

In [4]:
import torch

In [75]:
input_ids = torch.tensor([11,1,21,7]).unsqueeze(0)
target_ids = input_ids[:,1:]
logits = torch.randn(1,4,30)
log_probas = torch.log_softmax(logits,dim=-1)
next_tokens_log_probs = log_probas[:,:-1,:]
policy_log_probs = torch.gather(next_tokens_log_probs,dim=-1,index=target_ids.squeeze(0).unsqueeze(-1).unsqueeze(0)).squeeze(-1)
old_log_probs = policy_log_probs - 0.1

In [80]:
policy_log_probs ,old_log_probs

(tensor([[-3.2233, -2.6522, -2.7564]]), tensor([[-3.3233, -2.7522, -2.8564]]))

In [81]:
def ppo_policy_loss(policy_logps,old_logps,advantages,clip_eps=0.2):
    """
    policy_logps:   [B,T]
    old_logps:      [B,T]
    advantages:     [B,T]
    """
    ration = torch.exp(policy_logps-old_logps)
    r1 = ration * advantages #[B,T]
    r2 = torch.clamp(ration,1-clip_eps,1+clip_eps) * advantages
    loss = -torch.min(r1,r2).mean()
    return loss

tensor([[-3.3147, -4.0185, -4.3586]])

torch.Size([1, 3, 30])

In [ ]:
target_ids.squeeze(0).unsqueeze(-1).shape
.shape

torch.Size([1, 3, 1])

In [70]:
target_ids.unsqueeze(0)

tensor([[[ 1, 21,  7]]])